# Landing → Bronze: ocorrencias_rede

**Fonte:** `ocorrencias_rede.csv` (interrupções de fornecimento; cada registro é uma UC interrompida), separador vírgula, com cabeçalho.
**Destino:** `venergy.bronze.ocorrencias_rede` (Delta).

**Decisões desta camada**
- Conteúdo preservado: schema explícito com todas as colunas como texto (`STRING`), equivalente a `inferSchema = false`. Sem `trim`, limpeza, deduplicação ou conversão. O tratamento é feito na Silver.
- Contrato do cabeçalho: com `enforceSchema = false`, o Spark compara o cabeçalho do arquivo com `EXPECTED_COLUMNS`. Se a origem renomear ou reordenar colunas, a carga falha em vez de gravar dados deslocados.
- Linhas malformadas: `mode = FAILFAST`. Uma linha com número de colunas diferente do contrato interrompe a carga em vez de ser descartada ou completada com `null`.
- Colunas de rastreio: `source_file` (arquivo de origem) e `ingestion_datetime` (momento da carga), padrão do time.
- Carga completa com `overwrite`: o dataset é uma extração única, então reexecutar o notebook não duplica registros (o Workflow precisa ser reexecutável de ponta a ponta).
- Observação: o leitor CSV do Spark transforma campo vazio em `null`. Os demais marcadores de ausência em `causa` (`-`, `N/A`, `NULL`) chegam como texto, e `duracao_min` mantém o sufixo `" min"` quando ele existe no bruto.

As falhas encontradas estão em `docs/auditoria_dados/auditoria_ocorrencias_rede.md`.

In [ ]:
catalog = "venergy"
landing_schema = "landing"
bronze_schema = "bronze"
landing_volume = "landing_volume"
volume_path = f"/Volumes/{catalog}/{landing_schema}/{landing_volume}"

ocorrencias_path = f"{volume_path}/dados/ocorrencias_rede.csv"
bronze_table = f"{catalog}.{bronze_schema}.ocorrencias_rede"

EXPECTED_COLUMNS = [
    "id_ocorrencia",
    "alimentador",
    "id_uc",
    "tipo",
    "causa",
    "data_abertura",
    "data_encerramento",
    "duracao_min",
]
EXPECTED_ROWS = 180_000

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{landing_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{landing_schema}.{landing_volume}")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

schema_bruto = StructType([StructField(c, StringType(), True) for c in EXPECTED_COLUMNS])

df_ocorrencias = (
    spark.read
    .schema(schema_bruto)
    .option("header", True)
    .option("enforceSchema", False)
    .option("mode", "FAILFAST")
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .option("ignoreLeadingWhiteSpace", False)
    .option("ignoreTrailingWhiteSpace", False)
    .csv(ocorrencias_path)
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("ingestion_datetime", F.current_timestamp())
)

In [ ]:
(
    df_ocorrencias.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(bronze_table)
)

In [ ]:
# Contagem lida como texto puro (sem parser CSV), descontando o cabeçalho
linhas_arquivo = spark.read.text(ocorrencias_path).filter(F.length("value") > 0).count() - 1
tabela = spark.table(bronze_table)
linhas_tabela = tabela.count()

print({"linhas_arquivo": linhas_arquivo, "linhas_tabela": linhas_tabela, "esperado": EXPECTED_ROWS})
assert tabela.columns == EXPECTED_COLUMNS + ["source_file", "ingestion_datetime"], f"Colunas inesperadas: {tabela.columns}"
assert linhas_tabela == linhas_arquivo, "Contagem da tabela diverge do arquivo bruto"
assert linhas_tabela == EXPECTED_ROWS, "Contagem diverge do volume esperado da extração"
print("OK: 180.000 linhas carregadas em", bronze_table)

## Evidências das falhas (consultas sobre a Bronze)
As consultas abaixo reproduzem os principais achados do relatório de auditoria. Elas não alteram dados.

In [ ]:
t = bronze_table
duracao = "try_cast(regexp_replace(duracao_min, ' min$', '') AS INT)"
duracao_datas = "(unix_timestamp(data_encerramento) - unix_timestamp(data_abertura)) / 60"

# 1. Valores de causa, incluindo os marcadores de ausência
display(spark.sql(f"""
  SELECT coalesce(causa, '<null>') AS causa, COUNT(*) AS linhas
  FROM {t} GROUP BY 1 ORDER BY 2 DESC"""))

# 2. duracao_min: sufixo " min" e coerência com as datas
display(spark.sql(f"""
  SELECT COUNT_IF(duracao_min LIKE '% min') AS com_sufixo_min,
         COUNT_IF({duracao} IS NULL) AS nao_numericos_sem_sufixo,
         COUNT_IF({duracao} <> {duracao_datas}) AS divergentes_das_datas,
         COUNT_IF(data_encerramento < data_abertura) AS encerramento_antes_da_abertura,
         COUNT_IF({duracao} >= 1440) AS duracao_24h_ou_mais,
         MAX({duracao}) AS duracao_maxima_min
  FROM {t}"""))

# 3. Ocorrências por mês de abertura (pico em 2025-07)
display(spark.sql(f"""
  SELECT substr(data_abertura, 1, 7) AS mes, COUNT(*) AS ocorrencias
  FROM {t} GROUP BY 1 ORDER BY 1"""))

# 4. Concentração do pico de 2025-07 por alimentador e causa
display(spark.sql(f"""
  SELECT alimentador, coalesce(causa, '<null>') AS causa, COUNT(*) AS ocorrencias
  FROM {t} WHERE data_abertura LIKE '2025-07%'
  GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 10"""))

# 5. Interrupções sobrepostas na mesma UC (contam tempo em dobro no DEC se não forem unidas)
display(spark.sql(f"""
  SELECT substr(data_abertura, 1, 7) AS mes, COUNT(*) AS sobrepostas
  FROM (
    SELECT data_abertura,
           LAG(data_encerramento) OVER (PARTITION BY id_uc ORDER BY data_abertura, data_encerramento) AS fim_anterior
    FROM {t}
  )
  WHERE data_abertura < fim_anterior
  GROUP BY 1 ORDER BY 2 DESC"""))

# 6. Cobertura: período, alimentadores, municípios (prefixo do alimentador) e UCs
display(spark.sql(f"""
  SELECT MIN(data_abertura) AS primeira_abertura, MAX(data_abertura) AS ultima_abertura,
         COUNT(DISTINCT alimentador) AS alimentadores,
         COUNT(DISTINCT substr(alimentador, 5, 3)) AS prefixos_municipio,
         COUNT(DISTINCT id_uc) AS ucs_afetadas,
         COUNT(DISTINCT id_ocorrencia) AS ids_distintos
  FROM {t}"""))